# Joint parameter grid search — training partition only — zero transaction costs

Test **all 1,248 feasible combinations** jointly, with `trading_months <= formation_months`.
Formation: 3, 6, 12, 18 months; trading: 2, 3, 6, 12 months; return correlation: 0.8, 0.9;
Engle–Granger significance: 0.01, 0.05; entry: 1.5, 2.0, 2.5; exit: 0, 0.5;
stop: 3.0, 3.5, 4.0, 4.5.

All strategies trade from **2016-07-01** through the session preceding **2023-08-25**.
The last three years remain excluded at price ingestion. Trading windows are contiguous,
nonoverlapping, and the final window is shortened to the common end date.

The grid includes an **18-month formation period**. With data beginning in January 2015, that setting needs January 2015–June 2016 to select pairs before trading can begin in **July 2016**. I used July 2016 as the common trading start for every combination so performance is compared over identical market dates

The engine preserves notebook 03's log-price OLS models, daily log-return correlation
(strictly greater than the threshold), directional EG tests, next-close execution,
fixed holdings, equal allocation across selected pairs, proportional costs, and
stop blacklist until the next window. EG uses the returned **1%/5% critical value**,
consistent with the original 5% rule, rather than substituting an ordinary ADF test.

Formation diagnostics are cached by actual date interval. Each unit-capital pair
simulation is reused across correlation/EG settings with the same selected direction.
Capital scaling is exact for this engine's proportional costs and fractional holdings.
Two window schedules run concurrently by default. Completed combinations are skipped
when resuming; interrupted combinations are rebuilt.

Transaction costs: **0 bps per leg at entry and exit**. This zero-cost scenario recalculates holdings, equity, and performance with costs disabled. Outputs are separate from the 5 bps scenario; the historical Treasury benchmark still applies to Sharpe.


In [1]:
from pathlib import Path
import sys
import time
import pandas as pd
from IPython.display import display

PROJECT_FOLDER = Path.cwd()
if PROJECT_FOLDER.name == "notebooks":
    PROJECT_FOLDER = PROJECT_FOLDER.parent
assert (PROJECT_FOLDER / "src" / "joint_grid_search.py").is_file()
sys.path.insert(0, str(PROJECT_FOLDER / "src"))
from joint_grid_search import GridSearch, parameter_grid, PARAMETER_NAMES, Z_GRID

MAX_WORKERS = 2  # Increase only after checking CPU and memory usage.
TRANSACTION_COST_BPS = 0.0  # Zero transaction costs at entry and exit.
INITIAL_CAPITAL = 100_000.0
OUTPUT_ROOT = PROJECT_FOLDER / "outputs" / "joint_grid_search_zero_cost"
parameters = pd.DataFrame(parameter_grid())
assert len(parameters) == 1248
assert (parameters.trading_months <= parameters.formation_months).all()
display(parameters)
print(f"{len(parameters):,} combinations; {len(Z_GRID)} trading-rule combinations per selection setting")

,formation_months,trading_months,correlation_threshold,eg_significance,entry_z,exit_z,stop_z
0,3,2,0.8,0.01,1.5,0.0,3.0
1,3,2,0.8,0.01,1.5,0.0,3.5
2,3,2,0.8,0.01,1.5,0.0,4.0
3,3,2,0.8,0.01,1.5,0.0,4.5
4,3,2,0.8,0.01,1.5,0.5,3.0
...,...,...,...,...,...,...,...
1243,18,12,0.9,0.05,2.5,0.0,4.5
1244,18,12,0.9,0.05,2.5,0.5,3.0
1245,18,12,0.9,0.05,2.5,0.5,3.5
1246,18,12,0.9,0.05,2.5,0.5,4.0


1,248 combinations; 24 trading-rule combinations per selection setting


## Load training data and identify the resumable run

The output folder is identified by input file contents, engine source, package versions,
costs, and initial capital. Changing any of these creates a separate run, preventing stale
cache reuse. Loading and hashing read local files; only pre-holdout price observations
are interpreted or used. Do not run two notebook kernels into the same run folder at once.

In [2]:
search = GridSearch(PROJECT_FOLDER, output=OUTPUT_ROOT, cost_bps=TRANSACTION_COST_BPS,
                    initial_capital=INITIAL_CAPITAL).load_data()
print(f"Output folder: {search.folder}")
print(f"Training sessions: {len(search.prices):,}; stocks: {len(search.prices.columns):,}")
display(search.summary()[list(PARAMETER_NAMES) + ["Status"]].head())

Output folder: /home/lenovo/projects/pairs_trading_project/outputs/joint_grid_search_zero_cost/cb64c19613e1e11beb71
Training sessions: 2,176; stocks: 616


,formation_months,trading_months,correlation_threshold,eg_significance,entry_z,exit_z,stop_z,Status
0,3,2,0.8,0.01,1.5,0.0,3.0,pending
1,3,2,0.8,0.01,1.5,0.0,3.5,pending
2,3,2,0.8,0.01,1.5,0.0,4.0,pending
3,3,2,0.8,0.01,1.5,0.0,4.5,pending
4,3,2,0.8,0.01,1.5,0.5,3.0,pending


## Optional formation benchmark

This measures the first formation interval for representative 3- and 18-month windows.
It populates reusable caches and reports candidates requiring directional EG fits.
It is not an extrapolated full-search estimate: later candidate counts and simulation
work can differ substantially. Skip this cell to proceed directly to the search.

In [3]:
benchmark_rows = []
for formation_months in (3, 18):
    window = search.schedule(formation_months, 2).iloc[0]
    key = f"{window['Formation Start']:%Y%m%d}_{window['Formation End Exclusive']:%Y%m%d}"
    cache_hit = (search.cache_folder / f"{key}.pkl").exists()
    started = time.perf_counter()
    cached = search.formation(window)
    benchmark_rows.append({"formation_months": formation_months, "Cache Hit": cache_hit,
        "Seconds": time.perf_counter() - started,
        "Candidate Pairs": int(cached['comparisons']['Screen Passed'].sum()),
        "Directional Fits": len(cached['directions'])})
display(pd.DataFrame(benchmark_rows))

Cached formation 20160401_20160701: 954 directional fits
Cached formation 20150101_20160701: 424 directional fits


,formation_months,Cache Hit,Seconds,Candidate Pairs,Directional Fits
0,3,False,5.070586,477,954
1,18,False,5.293579,212,424


## Run or resume every feasible combination

This cell performs the full search and displays **one row for each of the 1,248 combinations**,
sorted by full-period Sharpe ratio. It saves each completed combination before proceeding.
The master summary is refreshed after each window-schedule batch. Failed/pending combinations
remain visible, including no-trade combinations. Inspect failures and rerun this cell to resume.

Each parameter folder contains `parameters.json`, `windows.csv`, `selected_pairs.csv`,
`trades.csv`, `portfolio_daily.csv`, `pair_daily.csv.gz`, `window_summary.csv`,
`summary.csv`, and `status.json`. Trades include signal/execution dates and z-scores,
entry/exit prices, signed holdings, hedge ratio, costs, realized P&L, holding duration,
and exit reason. The compressed daily audit records every selected pair's holdings
and account path, including pairs that never enter. Shared formation CSVs record
eligibility, correlations, both EG directions, and rejected-fit diagnostics.

`grid_summary.csv` contains the complete performance table. Runtime is reported for
the shared four-selection trading-rule batch, not as independent per-strategy timing.
Results use zero cash interest and the original engine's cost assumptions; borrow fees,
market impact, and integer-share constraints are not modeled. The gross-equity column
is the existing add-back-of-costs diagnostic, not a separately compounded costless run.

Sharpe is `(Annualized Return - Annualized Risk-Free Return) / Annualized Volatility`. The benchmark uses historical [FRED DGS3MO](https://fred.stlouisfed.org/series/DGS3MO) investment-basis three-month Treasury yields from `data/risk_free/DGS3MO.csv`. On each trading session, use the latest observation strictly before that date (at most seven calendar days old), accrue `(1 + yield / 100) ** (1 / 252) - 1`, compound over the exact evaluation dates, and annualize using 252 sessions/year. This is a rolling-bill yield proxy, not a Treasury total-return index. Window and full-period rates are calculated separately and exported as `Annualized Risk-Free Return`. Missing coverage halts execution; delete the CSV and restart the kernel to refresh. The Treasury data and benchmark implementation are fingerprinted, so old zero-risk-free results are not resumed.


In [4]:
grid_results = search.run(max_workers=MAX_WORKERS)
with pd.option_context("display.max_rows", len(grid_results), "display.max_columns", None):
    display(grid_results)
print(f"Summary saved to {search.folder / 'grid_summary.csv'}")
print(grid_results["Status"].value_counts().to_string())

Cached formation 20160701_20161001: 772 directional fits
Cached formation 20160601_20160901: 860 directional fits
Cached formation 20161001_20170101: 572 directional fits
Cached formation 20170101_20170401: 358 directional fits
Cached formation 20160801_20161101: 740 directional fits
Cached formation 20170401_20170701: 236 directional fits
Cached formation 20161201_20170301: 286 directional fits
Cached formation 20170701_20171001: 220 directional fits
Cached formation 20170201_20170501: 326 directional fits
Cached formation 20171001_20180101: 174 directional fits
Cached formation 20170601_20170901: 146 directional fits
Cached formation 20170801_20171101: 204 directional fits
Cached formation 20180101_20180401: 842 directional fits
Cached formation 20171201_20180301: 538 directional fits
Cached formation 20180401_20180701: 308 directional fits
Cached formation 20180701_20181001: 342 directional fits
Cached formation 20180201_20180501: 1238 directional fits
Cached formation 20180601_2018

,formation_months,trading_months,correlation_threshold,eg_significance,entry_z,exit_z,stop_z,Cumulative Return,Gross Cumulative Return,Annualized Return,Annualized Volatility,Annualized Risk-Free Return,Sharpe Ratio,Maximum Drawdown,Number of Trades,Win Rate,Average P/L per Trade ($),Average Return per Trade,Turnover (x),Transaction Costs ($),Costs / Initial Capital,Status,Windows,Trading Sessions,First Trading Date,Last Trading Date,Average Selected Pairs,Days With Exposure,Average Holding Sessions,Stop Exit Rate,Profitable Windows,Batch Runtime Seconds,Output Folder
0,18,6,0.8,0.01,1.5,0.5,4.0,0.186084,0.186084,0.024193,0.024839,0.015266,0.359417,0.034968,212,0.632075,87.775238,0.005466,56.333386,0.0,0.0,complete,15,1799,2016-07-01,2023-08-24,8.666667,1638,27.311321,0.254717,0.666667,20.698522,/home/lenovo/projects/pairs_trading_project/ou...
1,18,6,0.8,0.01,2.0,0.5,4.0,0.173055,0.173055,0.022610,0.021139,0.015266,0.347420,0.030069,153,0.562092,113.107996,0.005621,40.067308,0.0,0.0,complete,15,1799,2016-07-01,2023-08-24,8.666667,1502,28.620915,0.352941,0.733333,19.421113,/home/lenovo/projects/pairs_trading_project/ou...
2,18,6,0.8,0.01,2.0,0.0,4.0,0.175589,0.175589,0.022919,0.022350,0.015266,0.342437,0.036827,146,0.541096,120.266180,0.006705,38.173216,0.0,0.0,complete,15,1799,2016-07-01,2023-08-24,8.666667,1570,32.815068,0.369863,0.733333,20.173219,/home/lenovo/projects/pairs_trading_project/ou...
3,18,6,0.8,0.01,1.5,0.0,4.0,0.184455,0.184455,0.023996,0.026973,0.015266,0.323674,0.044017,191,0.602094,96.573302,0.006967,50.580141,0.0,0.0,complete,15,1799,2016-07-01,2023-08-24,8.666667,1691,33.727749,0.282723,0.666667,20.780264,/home/lenovo/projects/pairs_trading_project/ou...
4,6,2,0.9,0.05,1.5,0.0,4.5,0.178502,0.178502,0.023274,0.025562,0.015266,0.313275,0.031279,413,0.602906,43.220868,0.001778,136.817927,0.0,0.0,complete,43,1799,2016-07-01,2023-08-24,7.302326,1468,14.668281,0.193705,0.651163,55.439445,/home/lenovo/projects/pairs_trading_project/ou...
5,18,6,0.8,0.01,1.5,0.5,4.5,0.171546,0.171546,0.022425,0.025307,0.015266,0.282922,0.036767,215,0.627907,79.788801,0.005217,56.761161,0.0,0.0,complete,15,1799,2016-07-01,2023-08-24,8.666667,1653,29.572093,0.213953,0.666667,19.709129,/home/lenovo/projects/pairs_trading_project/ou...
6,18,6,0.8,0.01,2.5,0.5,4.0,0.149989,0.149989,0.019769,0.016424,0.015266,0.274200,0.024220,124,0.516129,120.958803,0.006777,31.539687,0.0,0.0,complete,15,1799,2016-07-01,2023-08-24,8.666667,1242,25.145161,0.435484,0.533333,17.803710,/home/lenovo/projects/pairs_trading_project/ou...
7,18,6,0.8,0.01,1.5,0.0,4.5,0.172112,0.172112,0.022495,0.027454,0.015266,0.263313,0.045125,194,0.603093,88.717437,0.006856,51.008269,0.0,0.0,complete,15,1799,2016-07-01,2023-08-24,8.666667,1694,36.226804,0.237113,0.666667,18.796035,/home/lenovo/projects/pairs_trading_project/ou...
8,18,6,0.8,0.01,2.0,0.0,4.5,0.161466,0.161466,0.021189,0.022744,0.015266,0.260423,0.036671,147,0.537415,109.841150,0.006388,38.362057,0.0,0.0,complete,15,1799,2016-07-01,2023-08-24,8.666667,1605,36.380952,0.312925,0.733333,20.226421,/home/lenovo/projects/pairs_trading_project/ou...
9,18,6,0.8,0.01,2.0,0.5,4.5,0.158912,0.158912,0.020874,0.021607,0.015266,0.259553,0.029912,154,0.558442,103.189383,0.005277,40.252124,0.0,0.0,complete,15,1799,2016-07-01,2023-08-24,8.666667,1568,32.006494,0.298701,0.666667,18.565213,/home/lenovo/projects/pairs_trading_project/ou...


Summary saved to /home/lenovo/projects/pairs_trading_project/outputs/joint_grid_search_zero_cost/cb64c19613e1e11beb71/grid_summary.csv
Status
complete    1248


## Reload saved results without rerunning

Use training results to investigate stable regions of neighboring parameters. Preserve the
holdout for final evaluation after choosing the strategy; this notebook never tests the holdout.

In [5]:
saved_results = search.summary()
display(saved_results)
print(f"Complete: {(saved_results['Status'] == 'complete').sum()} / {len(saved_results)}")

,formation_months,trading_months,correlation_threshold,eg_significance,entry_z,exit_z,stop_z,Cumulative Return,Gross Cumulative Return,Annualized Return,...,Trading Sessions,First Trading Date,Last Trading Date,Average Selected Pairs,Days With Exposure,Average Holding Sessions,Stop Exit Rate,Profitable Windows,Batch Runtime Seconds,Output Folder
0,18,6,0.8,0.01,1.5,0.5,4.0,0.186084,0.186084,0.024193,...,1799,2016-07-01,2023-08-24,8.666667,1638,27.311321,0.254717,0.666667,20.698522,/home/lenovo/projects/pairs_trading_project/ou...
1,18,6,0.8,0.01,2.0,0.5,4.0,0.173055,0.173055,0.022610,...,1799,2016-07-01,2023-08-24,8.666667,1502,28.620915,0.352941,0.733333,19.421113,/home/lenovo/projects/pairs_trading_project/ou...
2,18,6,0.8,0.01,2.0,0.0,4.0,0.175589,0.175589,0.022919,...,1799,2016-07-01,2023-08-24,8.666667,1570,32.815068,0.369863,0.733333,20.173219,/home/lenovo/projects/pairs_trading_project/ou...
3,18,6,0.8,0.01,1.5,0.0,4.0,0.184455,0.184455,0.023996,...,1799,2016-07-01,2023-08-24,8.666667,1691,33.727749,0.282723,0.666667,20.780264,/home/lenovo/projects/pairs_trading_project/ou...
4,6,2,0.9,0.05,1.5,0.0,4.5,0.178502,0.178502,0.023274,...,1799,2016-07-01,2023-08-24,7.302326,1468,14.668281,0.193705,0.651163,55.439445,/home/lenovo/projects/pairs_trading_project/ou...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,12,12,0.8,0.05,2.5,0.0,3.0,0.003356,0.003356,0.000469,...,1799,2016-07-01,2023-08-24,32.875000,1162,12.450549,0.824176,0.500000,9.099573,/home/lenovo/projects/pairs_trading_project/ou...
1244,12,12,0.8,0.01,2.5,0.0,3.0,-0.019051,-0.019051,-0.002691,...,1799,2016-07-01,2023-08-24,9.375000,484,9.400000,0.875000,0.125000,8.979644,/home/lenovo/projects/pairs_trading_project/ou...
1245,12,12,0.8,0.01,2.5,0.5,3.0,-0.017651,-0.017651,-0.002492,...,1799,2016-07-01,2023-08-24,9.375000,444,8.024096,0.843373,0.125000,9.594056,/home/lenovo/projects/pairs_trading_project/ou...
1246,18,12,0.8,0.05,2.5,0.0,3.0,-0.009961,-0.009961,-0.001401,...,1799,2016-07-01,2023-08-24,36.250000,1440,16.039216,0.847059,0.250000,6.067784,/home/lenovo/projects/pairs_trading_project/ou...


Complete: 1248 / 1248
